# VSPAERO mesh equalization

このNotebookでは、指定したreference Wingのchordwise tessellationを基準として、OpenVSPモデル全体のsurface mesh sizeを揃える。

`src/VSPAEROMesh.py`の`equalize_vspaero_tessellation()`が、各Geom / XSec sectionの実際の3次元surface lengthから`Tess_W` / `SectTess_U` / `Tess_U`を決め、OpenVSPが生成したmeshを再測定して補正する。このNotebookでは、入力条件の設定、公開関数の実行、結果表示だけを行う。

clustering parameterは変更しない。生成後のintersection mesh / NGon品質は、必要に応じて`mesh_quality/example_mesh_quality.ipynb`で別途確認する。


## 1. importとリポジトリ位置

Notebookの実行場所から親ディレクトリを順に調べ、`src/VSPAEROMesh.py`を含むリポジトリルートを見つける。


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

from IPython.display import display

start_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [start_dir, *start_dir.parents]
        if (candidate / "src" / "VSPAEROMesh.py").exists()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root containing src/VSPAEROMesh.py. "
        "Run this notebook inside the repository or set repo_root manually."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.VSPAEROMesh import equalize_vspaero_tessellation  # noqa: E402

print("repo_root:", repo_root)


## 2. 入力モデルとmesh条件

通常変更する設定はこのセルに集約する。`reference_tess_w`はreference Wingの`Tess_W`として設定され、そのWingで実際に生成されたchordwise 3D edge lengthが全機のtarget mesh sizeになる。

出力は元モデルとは別ファイルへ保存する。


In [ ]:
model_name = "Boeing_777-9x_mod"
# model_name = "G103A"
# model_name = "SampleGlider"

input_vsp3_path = (
    repo_root
    / "examples"
    / "models"
    / model_name
    / f"{model_name}.vsp3"
)
output_dir = (
    repo_root
    / "examples"
    / "notebooks"
    / "mesh_equalization"
    / f"results.{model_name}"
)
output_vsp3_path = output_dir / f"{model_name}.vsp3"

reference_wing_name = "WingGeom"
reference_tess_w = 17

# NoneならTess_Wを持つ全surface Geomを対象にする。
geom_names = None
tolerance = 0.10
max_iterations = 4

if not input_vsp3_path.exists():
    raise FileNotFoundError(input_vsp3_path)

output_dir.mkdir(parents=True, exist_ok=True)

print("input_vsp3_path :", input_vsp3_path)
print("output_vsp3_path:", output_vsp3_path)
print("reference_wing  :", reference_wing_name)
print("reference_tess_w:", reference_tess_w)


## 3. surface tessellationの均等化

mesh調整ロジックは`src/VSPAEROMesh.py`に置き、このNotebookでは公開関数を1回呼ぶ。元の`.vsp3`は変更せず、調整後モデルを`output_vsp3_path`へ保存する。


In [ ]:
summary = equalize_vspaero_tessellation(
    input_vsp3_path=input_vsp3_path,
    output_vsp3_path=output_vsp3_path,
    reference_wing_name=reference_wing_name,
    reference_tess_w=reference_tess_w,
    geom_names=geom_names,
    tolerance=tolerance,
    max_iterations=max_iterations,
)


## 4. 結果確認

`target_edge_size`はreference Wingから得た基準3D edge size。`after_section_u_edge_medians`、`after_u_edge_median`、`after_w_edge_median`、`after_edge_aspect_ratio`を見て、各Geomの物理mesh sizeがどこまで揃ったか確認する。


In [ ]:
summary_columns = [
    "geom_name",
    "geom_type",
    "status",
    "u_tessellation_mode",
    "target_edge_size",
    "before_tess_w",
    "after_tess_w",
    "before_u_tess",
    "after_u_tess",
    "after_section_u_edge_medians",
    "after_u_edge_median",
    "after_w_edge_median",
    "after_edge_aspect_ratio",
    "u_target_ratio",
    "w_target_ratio",
]

display(summary[summary_columns])
print("saved:", output_vsp3_path)
